# 📊 Step 4: Full-Scene Prediction, Accuracy Verification & Encroachment Report
### End-to-End Evaluation on Held-Out Test Blocks & Comprehensive Report for All Road Sections

**Outputs generated in this notebook:**
1. `building_probability.tif` and `building_coverage.tif` (full scene raster predictions).
2. `test_block_accuracy_report.csv` (Rigorous verification targeting ~0.85 F1/accuracy).
3. `encroachment_report_all_sections.csv` (Detailed metrics for **EVERY road section**).
4. `encroaching_buildings.gpkg` (All encroaching building polygon footprints with buffer zones and change status).
5. `road_sections_evaluated.gpkg` (Road section linestrings with encroachment statistics attached for GIS).

In [ ]:
# 1. Setup Environment & Load Libraries
import os, sys, joblib

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_DIR = "/content/drive/MyDrive/Encroachment"
    if not os.path.isdir(ROOT_DIR):
        ROOT_DIR = "/content/drive/My Drive/Encroachment"
    sys.path.append(ROOT_DIR)
    %pip install -q segmentation-models-pytorch rasterio geopandas lightgbm scipy shapely matplotlib
else:
    ROOT_DIR = os.getcwd()
    sys.path.append(ROOT_DIR)

import torch
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from rasterio.windows import Window

from encroach.config import Config
from encroach.scene import Scene
from encroach.models import build_model
from encroach.predict import predict_full_scene
from encroach.metrics import evaluate_test_blocks
from encroach.encroachment import generate_encroachment_report_all_sections

cfg = Config(ROOT=ROOT_DIR)
scene = Scene(cfg, load_state=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initialized Scene. Device:", device)

In [ ]:
# 2. Load Trained Deep Learning Model & LightGBM Model
dl_model = build_model(cfg).to(device)
checkpoint_path = cfg.P("best_unet_model.pth")
dl_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
dl_model.eval()
print("Loaded trained U-Net checkpoint from:", checkpoint_path)

gbm_path = cfg.P("gbm_classifier.joblib")
gbm_model = joblib.load(gbm_path) if os.path.exists(gbm_path) else None
if gbm_model is not None:
    print("Loaded trained LightGBM classifier for ensembling.")

In [ ]:
# 3. Run Full-Scene Prediction with 8-Way TTA & Gaussian Window Blending
# Set ensemble_weight to 1.0 for pure U-Net, or 0.65 to ensemble with LightGBM
predict_full_scene(
    scene=scene,
    dl_model=dl_model,
    device=device,
    gbm_model=gbm_model,
    ensemble_weight=0.70
)

In [ ]:
# 4. Rigorous Test-Block Accuracy Assessment (Target ~0.85)
# Evaluates strict pixel F1, 1-pixel tolerant F1, and per-building object detection rate
OPTIMAL_THR = 0.45  # Or auto-tuned threshold from validation set
metrics_df, det_df = evaluate_test_blocks(
    scene=scene,
    threshold=OPTIMAL_THR,
    hysteresis=cfg.HYSTERESIS,
    low_frac=cfg.HYST_LOW_FRAC
)

In [ ]:
# 5. Generate Encroachment Report for ALL Road Sections
# Cuts roads into 1000m sections, computes buffers (15m, 30m, 60m) & change detection
df_report, bld_encroaching = generate_encroachment_report_all_sections(
    scene=scene,
    segment_length=cfg.SEGMENT_M,
    buffer_zones=cfg.DIST_BANDS_M[:3]
)

print("Top 10 Most Encroached Road Sections:")
display(df_report.sort_values(by="total_encroaching_buildings", ascending=False).head(10))

In [ ]:
# 6. Visual Inspection Crop (High-Density Encroachment Area)
best_tile = None
max_blds = -1
cov_ds = rasterio.open(cfg.P("footprint_coverage.tif"))

for r0, c0, h, w in scene.TILES:
    t = scene.make_tile(r0, c0, h, w)
    n = int((cov_ds.read(1, window=t.cwin) >= cfg.BUILD_MIN_COVER * 100).sum())
    if n > max_blds:
        best_tile, max_blds = t, n
cov_ds.close()

t = best_tile
cr0 = t.r0 + max(0, (t.h - 600) // 2)
cc0 = t.c0 + max(0, (t.w - 600) // 2)
win = Window(cc0, cr0, min(600, t.w), min(600, t.h))

t_crop = scene.make_tile(cr0, cc0, win.height, win.width, halo=0)
t_crop = scene.load_imagery(t_crop)
img = t_crop.img
vv = (img > 0).all(0)

def norm_band(a):
    lo, hi = np.percentile(a[vv], (2, 98)) if vv.any() else (0, 1)
    return np.clip((a - lo) / (hi - lo + 1e-6), 0, 1)

fcc = np.dstack([norm_band(img[2]), norm_band(img[1]), norm_band(img[0])])
fcc[~vv] = 0

with rasterio.open(cfg.P("classification.tif")) as cd, rasterio.open(cfg.P("footprint_coverage.tif")) as covd:
    pred = cd.read(1, window=win)
    ref = covd.read(1, window=win) / 100.0

fig, ax = plt.subplots(1, 3, figsize=(21, 7), sharex=True, sharey=True)
ax[0].imshow(fcc); ax[0].set_title("LISS-4 False Colour Composite (NIR-R-G)")
ax[1].imshow(fcc); ax[1].imshow(np.where(ref >= cfg.BUILD_MIN_COVER, 1.0, np.nan), cmap="autumn", alpha=0.75)
ax[1].set_title("Ground Truth Reference Footprints (Red)")
ax[2].imshow(fcc); ax[2].imshow(np.where(pred == 1, 1.0, np.nan), cmap="spring", alpha=0.75)
ax[2].set_title("Deep Learning + Ensemble Predicted Encroachments")
for a in ax: a.axis("off")
plt.tight_layout()
plt.savefig(cfg.P("encroachment_result_overview.png"), dpi=150)
plt.show()
print("Saved visual comparison figure to:", cfg.P("encroachment_result_overview.png"))